# Downscaled `sfcWind` exploration

This notebook is for exploring downscaled surface wind speed (sfcWind) data. It was designed to work with the zarr outputs from the 4km ERA5-based CMIP6 downscaling effort.

The extreme threshold of ~31.3 m/s corresponds to 70 mph.

In [ ]:
# Set BASE_DIR environment variable to your path
# export BASE_DIR=/path/to/cmip6_4km_downscaling

# Pass parameters via command line arguments like so, for example:
# papermill downscaled_sfcWind.ipynb output.ipynb -p models 'EC-Earth3-Veg' -p scenarios 'historical ssp370 ssp585' -p threshold 31.3
models = "EC-Earth3-Veg"
scenarios = "historical ssp370 ssp585"
threshold = 31.3

In [ ]:
# Parameters
models = "EC-Earth3-Veg"
scenarios = "historical ssp126 ssp370 ssp585"
threshold = 31.3

In [ ]:
import math
import os
import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns
from pathlib import Path
from xclim.core.units import convert_units_to
import matplotlib.pyplot as plt
import gc
from baeda import projected_coords
from IPython.display import Markdown

import warnings
warnings.filterwarnings('ignore')

models = models.split(" ")
scenarios = scenarios.split(" ")

base_dir = Path(
    os.getenv("BASE_DIR", "/beegfs/CMIP6/crstephenson/EC-Earth3-Veg/cmip6_4km_downscaling")
)

ref_dir = base_dir.joinpath("era5_zarr")
cmip6_dir = base_dir.joinpath("cmip6_zarr")
adj_dir = base_dir.joinpath("adjusted")

random_points = []

In [ ]:
def open_sfcWind(model, scenario):
    zarr_store = adj_dir.joinpath(f"sfcWind_{model}_{scenario}_adjusted.zarr")
    ds = xr.open_zarr(zarr_store)
    sfcWind = ds.sfcWind  # units: m/s
    return sfcWind


def get_random_points(base, sfcWind_historical, n=5):
    if len(random_points) >= n:
        return random_points
    for _ in range(n):
        sfcWind_historical_extr = None
        while sfcWind_historical_extr is None or np.isnan(sfcWind_historical_extr).all():
            random_x = np.random.choice(base.x.values)
            random_y = np.random.choice(base.y.values)
            sfcWind_historical_extr = sfcWind_historical.sel(x=random_x, y=random_y)
        random_points.append((random_x, random_y))
    return random_points


def plot_adj_sfcWind(model, scenario, sfcWind):
    """Plot maps and histograms of adjusted (downscaled) data highlighting high wind pixels."""
    sfcWind_mean = sfcWind.mean("time")

    sfcWind_high = sfcWind > threshold
    sfcWind_high_count = sfcWind_high.sum("time")

    sfcWind_high_values = sfcWind.where(sfcWind_high).values.flatten()
    sfcWind_high_values = sfcWind_high_values[~np.isnan(sfcWind_high_values)]

    fig, axs = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f"High sfcWind analysis for {model}, {scenario}", fontsize=14)

    axs[0].imshow(
        sfcWind_mean.transpose("y", "x").values,
        cmap="Greys",
        alpha=0.5,
        interpolation="none",
    )

    masked_high = np.ma.masked_where(
        sfcWind_high_count.transpose("y", "x") == 0, sfcWind_high_count.transpose("y", "x")
    )
    im = axs[0].imshow(masked_high, cmap="Blues", alpha=0.8, interpolation="none")

    plt.colorbar(im, ax=axs[0], label=f"Count of days > {threshold} m/s (blue overlay)")
    axs[0].set_title(f"Mean sfcWind (grey) with days > {threshold} m/s (blue overlay)")
    axs[0].set_xlabel("x")
    axs[0].set_ylabel("y")

    if sfcWind_high_values.size > 0:
        axs[1].hist(sfcWind_high_values, bins=30, color="blue", alpha=0.7)
        axs[1].set_xlabel("sfcWind (m/s)")
        axs[1].set_ylabel("Frequency")
        axs[1].set_title(f"Histogram of daily sfcWind values > {threshold} m/s")
        percent_above_threshold = threshold * sfcWind_high_values.size / sfcWind.size
        axs[1].text(
            0.98,
            0.98,
            (
                f"{percent_above_threshold:.3f}% > {threshold} m/s"
                if np.round(percent_above_threshold, 3) > 0
                else f"~0% > {threshold} m/s"
            ),
            ha="right",
            va="top",
            transform=axs[1].transAxes,
            fontsize=12,
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"),
        )
    else:
        axs[1].text(
            0.5,
            0.5,
            f"No sfcWind values above {threshold} m/s found.",
            ha="center",
            va="center",
            fontsize=12,
        )
        axs[1].set_axis_off()

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.show()

    del sfcWind
    gc.collect()


def get_top6_high_pixels(sfcWind):
    sfcWind = sfcWind.transpose("y", "x", "time")
    sfcWind_high_count = (sfcWind > threshold).sum(dim="time").load()

    if np.sum(sfcWind_high_count.values > 0) < 6:
        return []

    top6_yx = np.unravel_index(np.argsort(sfcWind_high_count.values.ravel())[-6:], sfcWind_high_count.shape)

    top6_x = sfcWind_high_count.x.values[top6_yx[1]][::-1]
    top6_y = sfcWind_high_count.y.values[top6_yx[0]][::-1]

    return list(zip(top6_x, top6_y))


def plot_top6_high_pixels(model, scenario, sfcWind, top6_pixels):
    sfcWind_mean = sfcWind.mean("time")
    fig, ax = plt.subplots(1, 1, figsize=(9, 7))
    sfcWind_mean.transpose("y", "x").plot(ax=ax, cmap="Greys")
    top6_x, top6_y = zip(*top6_pixels)
    ax.scatter(top6_x, top6_y, color="blue", s=5, label=f"Top 6 high wind pixels (most occurrences of >{threshold} m/s)")
    plt.title(f"Mean sfcWind (grey) with top 6 pixels > {threshold} m/s (blue) for {model}, {scenario}", pad=20)
    plt.legend()
    plt.show()


def plot_top6_high_ecdfs(
    era5_ds, hist_ds, sim_ds, sfcWind_historical, sfcWind, top6_pixels, thresh=None, names=None
):
    n = len(top6_pixels)
    ncols = math.ceil(n / 2)
    nrows = 2

    fig, axs = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4), sharey=True)
    axs = axs.flatten()

    model = sim_ds.attrs.get("source_id")
    scenario = sim_ds.attrs.get("experiment_id")

    for i, (x, y) in enumerate(top6_pixels):
        era5_extr = era5_ds.wspd10_mean.sel(x=x, y=y, method="nearest").rename("sfcWind")
        hist_extr = hist_ds.sfcWind.sel(x=x, y=y, method="nearest")
        sim_extr = sim_ds.sfcWind.sel(x=x, y=y, method="nearest")
        sfcWind_historical_extr = sfcWind_historical.sel(x=x, y=y, method="nearest").load()
        sfcWind_extr = sfcWind.sel(x=x, y=y, method="nearest").load()

        max_doy = sfcWind_extr.time[sfcWind_extr.argmax(dim="time")].dt.dayofyear.item()
        doys = list(range(max_doy - 15, max_doy)) + list(range(max_doy, max_doy + 15))

        window_df = pd.concat(
            [
                era5_extr.sel(time=era5_extr.time.dt.dayofyear.isin(doys))
                .assign_coords(source="ERA5")
                .to_dataframe()
                .reset_index(),
                hist_extr.sel(time=hist_extr.time.dt.dayofyear.isin(doys))
                .rename("sfcWind")
                .assign_coords(source=f"{model}_historical")
                .to_dataframe()
                .reset_index(),
                sim_extr.sel(time=sim_extr.time.dt.dayofyear.isin(doys))
                .rename("sfcWind")
                .assign_coords(source=f"{model}_{scenario}")
                .to_dataframe()
                .reset_index(),
                sfcWind_historical_extr.sel(time=sfcWind_historical_extr.time.dt.dayofyear.isin(doys))
                .assign_coords(source=f"{model}_historical_downscaled")
                .to_dataframe()
                .reset_index(),
                sfcWind_extr.sel(time=sfcWind_extr.time.dt.dayofyear.isin(doys))
                .rename("sfcWind")
                .assign_coords(source=f"{model}_{scenario}_downscaled")
                .to_dataframe()
                .reset_index(),
            ]
        )[["time", "source", "sfcWind"]]

        window_df.reset_index(inplace=True)

        color_mapping = {
            "ERA5": "black",
            f"{model}_historical": "lightblue",
            f"{model}_{scenario}": "lightgreen",
            f"{model}_historical_downscaled": "blue",
            f"{model}_{scenario}_downscaled": "green",
        }
        sns.ecdfplot(
            data=window_df,
            x="sfcWind",
            hue="source",
            ax=axs[i],
            palette=color_mapping,
        )
        if thresh:
            axs[i].set_xlim(left=thresh)
        axs[i].set_title(f"Pixel {i+1}\n(x={x:.0f}, y={y:.0f})" if names is None else names[i])
        axs[i].set_xlabel("sfcWind (m/s)")
        axs[i].set_ylabel("ECDF" if i % ncols == 0 else "")

    for j in range(i + 1, nrows * ncols):
        fig.delaxes(axs[j])

    if names is None:
        suptitle = (
            f"ECDF of sfcWind for top 6 high wind pixels (most occurrences of >{threshold} m/s) for {model}.\n"
            f"Includes only data from 31-day window centered on the day-of-year (DOY: {max_doy}) of the maximum sfcWind value (all years)"
        )
    else:
        suptitle = (
            f"ECDF of sfcWind for 6 specified pixels for {model}.\n"
            f"Includes only data from 31-day window centered on the day-of-year (DOY: {max_doy}) of the maximum sfcWind value (all years)"
        )
    plt.suptitle(suptitle, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


def plot_doy_means(era5_ds, hist_ds, sim_ds, sfcWind_historical, sfcWind, x, y):
    sim_doy_mean = sim_ds["sfcWind"].sel(x=x, y=y).groupby("time.dayofyear").mean(dim="time")
    hist_doy_mean = hist_ds["sfcWind"].sel(x=x, y=y).groupby("time.dayofyear").mean(dim="time")
    ref_doy_mean = era5_ds["wspd10_mean"].sel(x=x, y=y).groupby("time.dayofyear").mean(dim="time")
    sfcWind_historical_doy_mean = sfcWind_historical.sel(x=x, y=y).groupby("time.dayofyear").mean(dim="time")
    sfcWind_doy_mean = sfcWind.sel(x=x, y=y).groupby("time.dayofyear").mean(dim="time")

    model = sim_ds.attrs.get("source_id")
    scenario = sim_ds.attrs.get("experiment_id")

    plt.figure(figsize=(9, 5))
    plt.plot(ref_doy_mean["dayofyear"], ref_doy_mean, label="ERA5", color="black")
    plt.plot(hist_doy_mean["dayofyear"], hist_doy_mean, label=f"{model} historical", color="lightblue")
    plt.plot(sim_doy_mean["dayofyear"], sim_doy_mean, label=f"{model} {scenario}", color="lightgreen")
    plt.plot(sfcWind_historical_doy_mean["dayofyear"], sfcWind_historical_doy_mean, label=f"{model} historical downscaled", color="blue")
    plt.plot(sfcWind_doy_mean["dayofyear"], sfcWind_doy_mean, label=f"{model} {scenario} downscaled", color="green")
    plt.xlabel("Day of Year")
    plt.ylabel("sfcWind (m/s)")
    plt.title(f"Day-of-year mean sfcWind: {model}, {scenario}, pixel: (x={x}, y={y})")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_random_ecdf(base, era5_ds, hist_ds, sim_ds, sfcWind_historical, sfcWind, ixy, thresh=None):
    """Plot ECDF for unadjusted and downscaled sfcWind at a point location."""
    model = sim_ds.attrs.get("source_id")
    scenario = sim_ds.attrs.get("experiment_id")

    fig, axs = plt.subplots(1, 2, figsize=(13, 6))
    fig.suptitle(
        f"sfcWind analysis for downscaled {model}, {scenario} at point location",
        fontsize=14,
    )

    base.plot(ax=axs[0], cmap="Greys", alpha=0.5, add_colorbar=False, x="x")

    random_x = ixy[0]
    random_y = ixy[1]

    axs[0].scatter(random_x, random_y, color="blue", s=5, label="Random Pixel")

    era5_extr = era5_ds.wspd10_mean.sel(x=random_x, y=random_y).rename("sfcWind")
    hist_extr = hist_ds.sfcWind.sel(x=random_x, y=random_y)
    sim_extr = sim_ds.sfcWind.sel(x=random_x, y=random_y)
    sfcWind_historical_extr = sfcWind_historical.sel(x=random_x, y=random_y).load()
    sfcWind_extr = sfcWind.sel(x=random_x, y=random_y).load()

    max_values = sfcWind_historical_extr.where(
        sfcWind_historical_extr == sfcWind_historical_extr.max(), drop=True
    )
    max_doy = max_values.time.dt.dayofyear.item()
    doys = list(range(max_doy - 15, max_doy)) + list(range(max_doy, max_doy + 15))

    window_df = pd.concat(
        [
            era5_extr.sel(time=era5_extr.time.dt.dayofyear.isin(doys))
            .assign_coords(source="ERA5")
            .to_dataframe()
            .reset_index(),
            hist_extr.sel(time=hist_extr.time.dt.dayofyear.isin(doys))
            .rename("sfcWind")
            .assign_coords(source=f"{model}_historical")
            .to_dataframe()
            .reset_index(),
            sim_extr.sel(time=sim_extr.time.dt.dayofyear.isin(doys))
            .rename("sfcWind")
            .assign_coords(source=f"{model}_{scenario}")
            .to_dataframe()
            .reset_index(),
            sfcWind_historical_extr.sel(time=sfcWind_historical_extr.time.dt.dayofyear.isin(doys))
            .assign_coords(source=f"{model}_historical_downscaled")
            .to_dataframe()
            .reset_index(),
            sfcWind_extr.sel(time=sfcWind_extr.time.dt.dayofyear.isin(doys))
            .rename("sfcWind")
            .assign_coords(source=f"{model}_{scenario}_downscaled")
            .to_dataframe()
            .reset_index(),
        ]
    )[["time", "source", "sfcWind"]]

    window_df.reset_index(inplace=True)

    color_mapping = {
        "ERA5": "black",
        f"{model}_historical": "lightblue",
        f"{model}_{scenario}": "lightgreen",
        f"{model}_historical_downscaled": "blue",
        f"{model}_{scenario}_downscaled": "green",
    }

    sns.ecdfplot(
        data=window_df,
        x="sfcWind",
        hue="source",
        ax=axs[1],
        palette=color_mapping,
    )
    if thresh:
        plt.xlim(left=thresh)
    axs[1].set_xlabel("sfcWind (m/s)")

    plt.tight_layout()
    plt.show()

    return window_df


def count_above_threshold(ds_var):
    return (ds_var > threshold).sum("time").load().sum().item()

In [ ]:
era5_ds = xr.open_dataset(ref_dir.joinpath("wspd10_mean_era5.zarr"))

### Choose hand-picked places

In [ ]:
places1 = ["Fairbanks", "Anchorage", "Nome", "Yakutat", "Utqiagvik", "near_McGrath"]
pixels1 = [(projected_coords[k]["x"], projected_coords[k]["y"]) for k in places1]

places2 = [
    "near_McGrath",
    "near_Arctic_Village",
    "warmest",
    "coldest",
    "wettest",
    "driest",
]
pixels2 = [(projected_coords[k]["x"], projected_coords[k]["y"]) for k in places2]

In [ ]:
for model in models:
    display(Markdown(f"# {model}"))
    hist_ds = xr.open_dataset(cmip6_dir.joinpath(f"sfcWind_{model}_historical.zarr"))
    sfcWind_historical = open_sfcWind(model, "historical")

    era5_count = count_above_threshold(era5_ds.wspd10_mean)
    historical_count = count_above_threshold(hist_ds.sfcWind)
    adj_historical_count = count_above_threshold(sfcWind_historical)

    for scenario in scenarios:
        display(Markdown(f"## {model}, {scenario}"))
        sfcWind = open_sfcWind(model, scenario)
        sim_ds = xr.open_dataset(cmip6_dir.joinpath(f"sfcWind_{model}_{scenario}.zarr"))

        sim_count = count_above_threshold(sim_ds.sfcWind)
        adj_count = count_above_threshold(sfcWind)

        top6_pixels = get_top6_high_pixels(sfcWind)
        if top6_pixels:
            display(Markdown(f"### High sfcWind analysis"))
            plot_adj_sfcWind(model, scenario, sfcWind)

            print(f"Counts of all pixels > {threshold} m/s across all days:")
            print(f"ERA5: {era5_count:,} pixels")
            if scenario != "historical":
                print(f"{model} historical (unadjusted): {historical_count:,} pixels")
                print(f"{model} historical (downscaled): {adj_historical_count:,} pixels")
            print(f"{model} {scenario} (unadjusted): {sim_count:,} pixels")
            print(f"{model} {scenario} (downscaled): {adj_count:,} pixels")

            display(Markdown(f"### Top 6 pixels with highest counts of sfcWind > {threshold} m/s"))
            print(f"Top 6 (x, y) coordinates with highest counts of days > {threshold} m/s:")
            for idx, (x, y) in enumerate(top6_pixels, start=1):
                print(
                    f"{idx}. (x: {x:.2f}, y: {y:.2f}), count above {threshold} m/s: {(sfcWind.sel(x=x, y=y).values > threshold).sum()}"
                )
            plot_top6_high_pixels(model, scenario, sfcWind, top6_pixels)
            if scenario != "historical":
                plot_top6_high_ecdfs(era5_ds, hist_ds, sim_ds, sfcWind_historical, sfcWind, top6_pixels)

                display(Markdown(f"### Day-of-year (DOY) means"))
                for i in range(6):
                    plot_doy_means(
                        era5_ds, hist_ds, sim_ds, sfcWind_historical, sfcWind, x=top6_pixels[i][0], y=top6_pixels[i][1]
                    )
        else:
            print(f"Fewer than 6 pixels found with sfcWind > {threshold} m/s.")

        if scenario != "historical":
            display(Markdown(f"### Hand-picked places"))
            plot_top6_high_ecdfs(era5_ds, hist_ds, sim_ds, sfcWind_historical, sfcWind, pixels1, names=places1)
            plot_top6_high_ecdfs(era5_ds, hist_ds, sim_ds, sfcWind_historical, sfcWind, pixels2, names=places2)

            display(Markdown(f"### Random locations"))
            base = sfcWind.max("time")
            random_points = get_random_points(base, sfcWind_historical)
            for ixy in random_points:
                plot_random_ecdf(base, era5_ds, hist_ds, sim_ds, sfcWind_historical, sfcWind, ixy)